In [ ]:
from google.colab import drive
import os

# 1. Menghubungkan ke Google Drive (Nanti akan muncul pop-up untuk izin akses)
drive.mount('/content/drive')

# 2. Membuat folder khusus untuk proyek ini di Drive
project_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait'

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Folder berhasil dibuat di: {project_path}")
else:
    print(f"Folder sudah ada di: {project_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder sudah ada di: /content/drive/MyDrive/Proyek_NLP_Clickbait


In [ ]:
# Install kagglehub terlebih dahulu (karena bawaan Colab kadang belum ada versi terbarunya)
!pip install kagglehub -q

import kagglehub
import shutil

print("Mulai mengunduh dataset dari Kaggle...")
# Download dataset ke penyimpanan sementara Colab
path_temp = kagglehub.dataset_download("andikawilliam/clickid")

# Menentukan lokasi penyimpanan di Google Drive kita
dataset_drive_path = f"{project_path}/dataset_clickid"

# Memindahkan data ke Google Drive
if not os.path.exists(dataset_drive_path):
    shutil.copytree(path_temp, dataset_drive_path)
    print(f"Dataset berhasil dipindahkan secara permanen ke Drive: {dataset_drive_path}")
else:
    print(f"Dataset sudah tersimpan aman di Drive: {dataset_drive_path}")

Mulai mengunduh dataset dari Kaggle...
Using Colab cache for faster access to the 'clickid' dataset.
Dataset sudah tersimpan aman di Drive: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid


In [ ]:
# Melihat daftar file yang ada di dalam folder dataset
isi_folder = os.listdir(dataset_drive_path)
print("Daftar file di dalam folder dataset:")
for file in isi_folder:
    print(f"- {file}")

Daftar file di dalam folder dataset:
- annotated
- raw
- dataset_features_final.csv


In [ ]:
import pandas as pd
import os

dataset_drive_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid'

print("Mencari file CSV di dalam subfolder...")
csv_files = []

# os.walk akan menelusuri semua folder dan subfolder secara otomatis
for root, dirs, files in os.walk(dataset_drive_path):
    for file in files:
        if file.endswith('.csv'):
            full_path = os.path.join(root, file)
            csv_files.append(full_path)
            print(f"Ditemukan file: {full_path}")

if len(csv_files) > 0:
    # Kita baca file CSV pertama yang ditemukan di dalam folder 'annotated'
    # Biasanya data yang sudah dianotasi (diberi label) yang siap dipakai untuk supervised learning
    file_pilihan = csv_files[0]
    for f in csv_files:
        if 'annotated' in f:
            file_pilihan = f
            break

    print(f"\nMembaca data dari: {file_pilihan}")
    df = pd.read_csv(file_pilihan)

    print(f"Total baris: {df.shape[0]}, Total kolom: {df.shape[1]}\n")
    display(df.head())
else:
    print("\nFile CSV masih tidak ditemukan. Coba kita cek manual format filenya.")

Mencari file CSV di dalam subfolder...
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/dataset_features_final.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_tribunnews.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_republika.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_sindonews.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_detikNews.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_tempo.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annoated_pos_metro.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/csv/annotated_kompas.csv
Ditemukan file: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset

,title,label,label_score
0,Prediksi Susunan Pemain Timnas Inggris vs Koso...,non-clickbait,0
1,"Ambil Formulir Pilkada Tangsel dari PDI-P, Sit...",non-clickbait,0
2,Mulan Jameela Mantan Duet Maia Estianty Rela M...,clickbait,1
3,Dukung Operasional Bandara Internasional Kerta...,non-clickbait,0
4,"KPK Cekal Melchias Markus Mekeng, Golkar Minta...",non-clickbait,0


In [ ]:
import re

def basic_clean(text):
    # 1. Mengubah ke huruf kecil
    text = str(text).lower()
    # 2. Menghapus spasi berlebih di tengah dan ujung kalimat
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Menerapkan pembersihan dasar pada kolom 'title'
df['clean_title'] = df['title'].apply(basic_clean)

# Menampilkan sampel hasil pembersihan dasar
print("Hasil Pembersihan Dasar (Lowercasing & Spasi):")
display(df[['title', 'clean_title']].head())

Hasil Pembersihan Dasar (Lowercasing & Spasi):


,title,clean_title
0,Prediksi Susunan Pemain Timnas Inggris vs Koso...,prediksi susunan pemain timnas inggris vs koso...
1,"Ambil Formulir Pilkada Tangsel dari PDI-P, Sit...","ambil formulir pilkada tangsel dari pdi-p, sit..."
2,Mulan Jameela Mantan Duet Maia Estianty Rela M...,mulan jameela mantan duet maia estianty rela m...
3,Dukung Operasional Bandara Internasional Kerta...,dukung operasional bandara internasional kerta...
4,"KPK Cekal Melchias Markus Mekeng, Golkar Minta...","kpk cekal melchias markus mekeng, golkar minta..."


In [ ]:
def extract_clickbait_signatures(row):
    text = row['clean_title']

    # Menghitung elemen khas clickbait
    seru_count = text.count('!')
    tanya_count = text.count('?')
    elipsis_count = len(re.findall(r'\.\.\.', text))

    # Memeriksa apakah ada angka di dalam judul (pola listicle)
    digits = re.findall(r'\d+', text)
    digit_count = len(digits)

    return pd.Series([seru_count, tanya_count, elipsis_count, digit_count])

# Membuat kolom fitur baru berdasarkan karakteristik clickbait
df[['jml_tanda_seru', 'jml_tanda_tanya', 'jml_elipsis', 'jumlah_angka']] = df.apply(extract_clickbait_signatures, axis=1)

print("Hasil Ekstraksi Karakteristik Clickbait:")
display(df[['clean_title', 'jml_tanda_seru', 'jml_tanda_tanya', 'jml_elipsis', 'jumlah_angka']].head())

Hasil Ekstraksi Karakteristik Clickbait:


,clean_title,jml_tanda_seru,jml_tanda_tanya,jml_elipsis,jumlah_angka
0,prediksi susunan pemain timnas inggris vs koso...,0,0,0,0
1,"ambil formulir pilkada tangsel dari pdi-p, sit...",0,0,0,0
2,mulan jameela mantan duet maia estianty rela m...,0,1,0,0
3,dukung operasional bandara internasional kerta...,0,0,0,0
4,"kpk cekal melchias markus mekeng, golkar minta...",0,0,0,0


In [ ]:
def remove_symbols(text):
    # Menghapus semua karakter selain huruf alfabet dan spasi
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Merapikan kembali spasi setelah penghapusan simbol
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Membersihkan teks dari simbol untuk persiapan tokenisasi
df['text_only'] = df['clean_title'].apply(remove_symbols)

print("Hasil Pembersihan Karakter Non-Esensial:")
display(df[['clean_title', 'text_only']].head())

Hasil Pembersihan Karakter Non-Esensial:


,clean_title,text_only
0,prediksi susunan pemain timnas inggris vs koso...,prediksi susunan pemain timnas inggris vs koso...
1,"ambil formulir pilkada tangsel dari pdi-p, sit...",ambil formulir pilkada tangsel dari pdi p siti...
2,mulan jameela mantan duet maia estianty rela m...,mulan jameela mantan duet maia estianty rela m...
3,dukung operasional bandara internasional kerta...,dukung operasional bandara internasional kerta...
4,"kpk cekal melchias markus mekeng, golkar minta...",kpk cekal melchias markus mekeng golkar minta ...


In [ ]:
def tokenize_text(text):
    # Memecah kalimat menjadi list kata berdasarkan spasi
    return text.split()

# Menerapkan tokenisasi kata
df['tokens'] = df['text_only'].apply(tokenize_text)

print("Hasil Word Tokenization (Pecah Kata):")
display(df[['text_only', 'tokens']].head())

Hasil Word Tokenization (Pecah Kata):


,text_only,tokens
0,prediksi susunan pemain timnas inggris vs koso...,"[prediksi, susunan, pemain, timnas, inggris, v..."
1,ambil formulir pilkada tangsel dari pdi p siti...,"[ambil, formulir, pilkada, tangsel, dari, pdi,..."
2,mulan jameela mantan duet maia estianty rela m...,"[mulan, jameela, mantan, duet, maia, estianty,..."
3,dukung operasional bandara internasional kerta...,"[dukung, operasional, bandara, internasional, ..."
4,kpk cekal melchias markus mekeng golkar minta ...,"[kpk, cekal, melchias, markus, mekeng, golkar,..."


In [ ]:
# Mengunduh dan menginstal library Sastrawi di sesi Colab saat ini
!pip install Sastrawi -q

import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengambil stopword bawaan Sastrawi
factory = StopWordRemoverFactory()
sastrawi_stopwords = factory.get_stop_words()

# Kustomisasi: Mengeluarkan kata ganti penunjuk penting dari daftar hapus clickbait
clickbait_triggers = ['ini', 'itu', 'begini', 'begitu', 'dia', 'mereka']
custom_stopwords = [word for word in sastrawi_stopwords if word not in clickbait_triggers]

def remove_stopwords_from_list(token_list):
    # Hanya mempertahankan kata yang tidak terdaftar di custom_stopwords
    # Pastikan data yang masuk adalah list (jika bukan, kembalikan list kosong atau list awal)
    if isinstance(token_list, list):
        return [word for word in token_list if word not in custom_stopwords]
    return token_list

# Menerapkan penyaringan kata hubung selektif
df['filtered_tokens'] = df['tokens'].apply(remove_stopwords_from_list)

print("Hasil Penyaringan Kata Hubung (Stopword Removal):")
display(df[['tokens', 'filtered_tokens']].head())

Hasil Penyaringan Kata Hubung (Stopword Removal):


,tokens,filtered_tokens
0,"[prediksi, susunan, pemain, timnas, inggris, v...","[prediksi, susunan, pemain, timnas, inggris, v..."
1,"[ambil, formulir, pilkada, tangsel, dari, pdi,...","[ambil, formulir, pilkada, tangsel, pdi, p, si..."
2,"[mulan, jameela, mantan, duet, maia, estianty,...","[mulan, jameela, mantan, duet, maia, estianty,..."
3,"[dukung, operasional, bandara, internasional, ...","[dukung, operasional, bandara, internasional, ..."
4,"[kpk, cekal, melchias, markus, mekeng, golkar,...","[kpk, cekal, melchias, markus, mekeng, golkar,..."


In [ ]:
# Menyimpan dataframe final hasil Tahap 1-5 ke Google Drive
final_save_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/dataset_features_final.csv'
df.to_csv(final_save_path, index=False)

print(f"Data preprocessing selesai diamankan di: {final_save_path}")

Data preprocessing selesai diamankan di: /content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/dataset_features_final.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("1. Memuat kembali data yang sudah bersih...")
file_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/dataset_features_final.csv'

# Membaca data dan membuang baris yang kosong
df = pd.read_csv(file_path).dropna(subset=['filtered_tokens'])

# Menggabungkan kembali token list menjadi string kalimat utuh untuk TF-IDF
df['final_text'] = df['filtered_tokens'].apply(lambda x: " ".join(eval(x)) if isinstance(x, str) else "")

print("2. Membagi Data (Train 80% & Test 20%)...")
X = df['final_text']
y = df['label']

# Membagi data secara proporsional
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Jumlah Data Latih: {X_train.shape[0]}")
print(f"Jumlah Data Uji: {X_test.shape[0]}")

print("\n3. Membangun Vektor TF-IDF dengan N-Grams...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Matriks fitur TF-IDF berhasil dibuat!")
print(f"Bentuk Matriks Training: {X_train_tfidf.shape}")

1. Memuat kembali data yang sudah bersih...
2. Membagi Data (Train 80% & Test 20%)...
Jumlah Data Latih: 1200
Jumlah Data Uji: 300

3. Membangun Vektor TF-IDF dengan N-Grams...
Matriks fitur TF-IDF berhasil dibuat!
Bentuk Matriks Training: (1200, 5000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("1. Melatih Model Machine Learning (Logistic Regression)...")
# Menginisialisasi algoritma
model = LogisticRegression(random_state=42, max_iter=1000)

# Proses 'Belajar': Memasukkan matriks fitur dan kunci jawaban (label)
model.fit(X_train_tfidf, y_train)
print("Model selesai dilatih!\n")

print("2. Melakukan Prediksi pada Data Uji (Test)...")
# Meminta model menebak apakah 300 judul berita baru ini clickbait atau bukan
y_pred = model.predict(X_test_tfidf)

print("3. Evaluasi Hasil Prediksi (Rapor Model):")
# Menghitung seberapa banyak tebakan yang benar
akurasi = accuracy_score(y_test, y_pred)
print(f"Akurasi Keseluruhan: {akurasi * 100:.2f}%\n")

# Menampilkan detail laporan klasifikasi (Precision, Recall, F1-Score)
print("Detail Laporan Klasifikasi:")
print(classification_report(y_test, y_pred))

1. Melatih Model Machine Learning (Logistic Regression)...
Model selesai dilatih!

2. Melakukan Prediksi pada Data Uji (Test)...
3. Evaluasi Hasil Prediksi (Rapor Model):
Akurasi Keseluruhan: 75.00%

Detail Laporan Klasifikasi:
               precision    recall  f1-score   support

    clickbait       0.74      0.99      0.85       210
non-clickbait       0.89      0.19      0.31        90

     accuracy                           0.75       300
    macro avg       0.82      0.59      0.58       300
 weighted avg       0.79      0.75      0.69       300



Percobaan Deep Learning

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from sklearn.model_selection import train_test_split

print("1. Memuat FULL DATASET (15.000 baris)...")
file_utama = '/content/drive/MyDrive/Proyek_NLP_Clickbait/dataset_clickid/annotated/combined/csv/main.csv'
df = pd.read_csv(file_utama)

# Memastikan teks menjadi string kecil (tanpa membuang stopword agar konteks kalimat terjaga untuk LSTM)
df['teks_bersih'] = df['title'].astype(str).str.lower()

print("2. Proses Tokenisasi (Mengubah kata menjadi urutan angka)...")
# Mengatur batas kosakata maksimal dan panjang kalimat
vocab_size = 10000
max_length = 30 # Rata-rata judul berita tidak lebih dari 30 kata
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

# Melatih token pada data teks
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(df['teks_bersih'])

# Mengubah teks menjadi sekuens angka dan menyamakan panjangnya (Padding)
sequences = tokenizer.texts_to_sequences(df['teks_bersih'])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

print("3. Membagi Data (Train 80% / Test 20%)...")
# Label: Ubah 'clickbait' jadi 1, 'non-clickbait' jadi 0
labels = np.array(df['label'].apply(lambda x: 1 if x == 'clickbait' else 0))

X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels)

print("4. Merakit Arsitektur Deep Learning (LSTM)...")
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    # Bidirectional LSTM membaca kalimat dari kiri ke kanan dan kanan ke kiri secara bersamaan
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3), # Mencegah model menghafal jawaban (overfitting)
    Bidirectional(LSTM(32)),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Lapisan output untuk menebak 1 (Clickbait) atau 0 (Faktual)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

print("5. Mulai Melatih Mesin (Proses memakan waktu sekitar 3-5 menit)...")
# Melatih mesin selama 5 putaran (epochs)
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

print("\nPelatihan Selesai! Cek angka 'val_accuracy' di baris terakhir untuk melihat rapor akhir modelmu.")

1. Memuat FULL DATASET (15.000 baris)...
2. Proses Tokenisasi (Mengubah kata menjadi urutan angka)...
3. Membagi Data (Train 80% / Test 20%)...
4. Merakit Arsitektur Deep Learning (LSTM)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

5. Mulai Melatih Mesin (Proses memakan waktu sekitar 3-5 menit)...
Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 46s 80ms/step - accuracy: 0.7307 - loss: 0.5428 - val_accuracy: 0.7813 - val_loss: 0.4879
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 40s 77ms/step - accuracy: 0.8499 - loss: 0.3675 - val_accuracy: 0.7707 - val_loss: 0.4956
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 40s 72ms/step - accuracy: 0.8994 - loss: 0.2631 - val_accuracy: 0.7577 - val_loss: 0.5681
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 28s 75ms/step - accuracy: 0.9417 - loss: 0.1680 - val_accuracy: 0.7477 - val_loss: 0.8257
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 28s 76ms/step - accuracy: 0.9672 - loss: 0.0963 - val_accuracy: 0.7460 - val_loss: 1.0583

Pelatihan Selesai! Cek angka 'val_accuracy' di baris terakhir untuk melihat rapor akhir modelmu.


In [ ]:
import numpy as np

print("--- TESTING MODEL LSTM DENGAN JUDUL BARU ---")

# 1. Menyiapkan beberapa judul berita baru untuk diuji
judul_tes = [
    "Astaga! Artis Ini Ketahuan Makan Nasi Padang Pakai Tangan Kiri, Bikin Netizen Melongo!", # Contoh jelas Clickbait
    "Pemerintah Provinsi Riau Menetapkan UMP Tahun 2026 Sebesar 3.5 Juta Rupiah", # Contoh Faktual / Berita Resmi
    "5 Rahasia Cepat Kaya Tanpa Kerja yang Disembunyikan Para Miliarder" # Contoh pola Listicle (Clickbait)
]

# 2. Preprocessing Data Baru
# Menyeragamkan ke huruf kecil
judul_tes_bersih = [teks.lower() for teks in judul_tes]

# WAJIB: Mengubah teks menjadi angka menggunakan 'tokenizer' yang sudah dilatih di cell sebelumnya
tes_sequences = tokenizer.texts_to_sequences(judul_tes_bersih)

# WAJIB: Menyamakan panjang kalimat (Padding) menjadi 30 kata
tes_padded = pad_sequences(tes_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# 3. Eksekusi Prediksi
print("Sedang memproses prediksi...")
prediksi = model.predict(tes_padded)

# 4. Membaca dan Menampilkan Hasil Prediksi
for i in range(len(judul_tes)):
    print("-" * 50)
    print(f"Judul Berita : '{judul_tes[i]}'")

    # Model 'sigmoid' menghasilkan probabilitas dari 0.0 hingga 1.0
    probabilitas = prediksi[i][0]

    # Logika klasifikasi: Jika probabilitas di atas 50%, maka itu Clickbait
    if probabilitas >= 0.5:
        label = "CLICKBAIT 🔴"
        keyakinan = probabilitas * 100
    else:
        label = "FAKTUAL (NON-CLICKBAIT) 🟢"
        keyakinan = (1 - probabilitas) * 100 # Dibalik agar persentasenya relevan dengan kelas 0

    print(f"Prediksi     : {label}")
    print(f"Tingkat Yakin: {keyakinan:.2f}%")

--- TESTING MODEL LSTM DENGAN JUDUL BARU ---
Sedang memproses prediksi...
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
--------------------------------------------------
Judul Berita : 'Astaga! Artis Ini Ketahuan Makan Nasi Padang Pakai Tangan Kiri, Bikin Netizen Melongo!'
Prediksi     : CLICKBAIT 🔴
Tingkat Yakin: 99.99%
--------------------------------------------------
Judul Berita : 'Pemerintah Provinsi Riau Menetapkan UMP Tahun 2026 Sebesar 3.5 Juta Rupiah'
Prediksi     : FAKTUAL (NON-CLICKBAIT) 🟢
Tingkat Yakin: 85.21%
--------------------------------------------------
Judul Berita : '5 Rahasia Cepat Kaya Tanpa Kerja yang Disembunyikan Para Miliarder'
Prediksi     : CLICKBAIT 🔴
Tingkat Yakin: 99.94%


In [ ]:
import pickle

print("Menyimpan aset model Deep Learning...")

# 1. Menyimpan Arsitektur dan Bobot Model LSTM (Format H5)
model_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait/lstm_clickbait_model.h5'
model.save(model_path)
print(f"✅ Model LSTM berhasil disimpan di: {model_path}")

# 2. Menyimpan Kamus Kata (Tokenizer) agar web bisa menerjemahkan teks baru menjadi angka
tokenizer_path = '/content/drive/MyDrive/Proyek_NLP_Clickbait/tokenizer.pickle'
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"✅ Tokenizer berhasil disimpan di: {tokenizer_path}")

print("\nSelamat! Seluruh pipeline Preprocessing dan Modelling telah selesai!")

Menyimpan aset model Deep Learning...
✅ Model LSTM berhasil disimpan di: /content/drive/MyDrive/Proyek_NLP_Clickbait/lstm_clickbait_model.h5
✅ Tokenizer berhasil disimpan di: /content/drive/MyDrive/Proyek_NLP_Clickbait/tokenizer.pickle

Selamat! Seluruh pipeline Preprocessing dan Modelling telah selesai!
